# Asistente virtual de Seoul Glow
Chatbot basado en reglas (NLTK) para la boutique en linea de skincare coreano **Seoul Glow**.

Avance de proyecto 1 - Desarrollo de aplicaciones con inteligencia artificial.

Autora: Iris Bringas (matricula AL07302962)


## 1. Preparacion
Se usan dos piezas de NLTK que no requieren descargar un corpus (`RegexpTokenizer` y `SnowballStemmer`), porque la descarga automatica de NLTK (`punkt_tab`, `stopwords`) no estaba disponible en el entorno donde desarrolle y probe este cuaderno. La lista de stopwords en espaniol se define a mano.

In [1]:
import re
import unicodedata
import random
import pandas as pd
from nltk.tokenize import RegexpTokenizer
from nltk.stem import SnowballStemmer

_tokenizer = RegexpTokenizer(r"[a-zA-Z0-9\u00e1\u00e9\u00ed\u00f3\u00fa\u00f1]+")
_stemmer = SnowballStemmer("spanish")

STOPWORDS_ES = {
    "a", "al", "algo", "algunas", "algunos", "ante", "antes", "como", "con",
    "contra", "cual", "cuando", "de", "del", "desde", "donde", "durante",
    "e", "el", "ella", "ellas", "ellos", "en", "entre", "era", "erais",
    "eramos", "eran", "eres", "es", "esa", "esas", "ese", "eso", "esos",
    "esta", "estaba", "estado", "estamos", "estan", "estar", "este",
    "esto", "estos", "fue", "fueron", "ha", "hace", "hacia", "han", "hay",
    "la", "las", "le", "les", "lo", "los", "me", "mi", "mis", "mucho",
    "muy", "nos", "nosotros", "o", "os", "otra", "otro", "para", "pero",
    "poco", "por", "porque", "que", "quien", "se", "ser", "si", "sin",
    "sobre", "soy", "su", "sus", "te", "tu", "tus", "un", "una", "uno",
    "unos", "y", "ya", "yo",
}

print("Librerias listas.")


Librerias listas.


## 2. Base de conocimiento
Intenciones con sus frases de ejemplo y respuestas, mas las entidades del negocio (tipo de piel, producto, ingrediente) y los datos reales (precios, rutinas, informacion de ingredientes).

In [2]:
INTENCIONES = {
    "saludo": {
        "ejemplos": ["hola", "buenas", "buenos dias", "buenas tardes", "buenas noches",
                     "que tal", "hola buen dia", "hey", "oli"],
        "respuestas": ["Hola, soy el asistente virtual de Seoul Glow. Puedo ayudarte "
                       "con rutinas, ingredientes, precios, envios y cambios. "
                       "En que te ayudo?"]},
    "consultar_rutina": {
        "ejemplos": ["que rutina me recomiendan", "que rutina es buena para mi piel",
                      "como armo mi rutina de skincare", "que productos debo usar",
                      "rutina para piel grasa", "rutina para piel seca",
                      "rutina para piel sensible", "que pasos debo seguir en mi rutina",
                      "recomiendame una rutina", "cual es el orden correcto de mis productos"]},
    "consultar_ingredientes": {
        "ejemplos": ["que hace la niacinamida", "para que sirve el acido hialuronico",
                      "que es la centella asiatica", "sirve la mucina de caracol",
                      "que beneficios tiene el retinol", "la vitamina c mancha la piel",
                      "puedo usar retinol y vitamina c juntos", "que ingrediente ayuda con las manchas",
                      "el acido hialuronico reseca la piel"]},
    "consultar_precio": {
        "ejemplos": ["cuanto cuesta el limpiador", "precio del serum",
                      "cuanto sale la mascarilla", "que precio tiene el protector solar",
                      "cuanto cuesta el tonico", "precio de la crema", "cuanto vale el kit completo"]},
    "consultar_envio": {
        "ejemplos": ["cuanto tarda el envio", "cuando llega mi pedido",
                      "hacen envios a todo mexico", "cuanto cuesta el envio",
                      "puedo rastrear mi pedido", "hacen envios internacionales", "tienen envio gratis"]},
    "soporte_posventa": {
        "ejemplos": ["quiero cambiar un producto", "como hago una devolucion",
                      "mi pedido llego incompleto", "quiero cancelar mi pedido",
                      "mi pedido nunca llego", "llego daniado el empaque", "cual es su politica de cambios"]},
    "despedida": {
        "ejemplos": ["gracias hasta luego", "adios", "nos vemos",
                      "muchas gracias eso es todo", "listo gracias", "bye"],
        "respuestas": ["Gracias por escribir a Seoul Glow. Que tengas un dia con mucho glow."]},
}

FALLBACK = ("No estoy segura de haber entendido bien. Puedo ayudarte con rutinas, "
            "ingredientes, precios, envios o cambios/devoluciones. Sobre cual "
            "quieres que te ayude?")

PALABRAS_ALERTA = ["reaccion", "irrit", "ardor", "alergi", "rojez", "hinchaz",
                    "comezon", "picazon", "me quemo", "se me quemo"]

MENSAJE_ALERTA = ("Lamento escuchar eso. Por tu seguridad suspende el uso del producto "
                   "de inmediato, enjuaga la zona con agua tibia y evita el sol directo. "
                   "Te voy a canalizar con una persona del equipo para dar seguimiento a "
                   "tu caso; esto no sustituye una valoracion con un dermatologo.")

print("Intenciones cargadas:", list(INTENCIONES.keys()))


Intenciones cargadas: ['saludo', 'consultar_rutina', 'consultar_ingredientes', 'consultar_precio', 'consultar_envio', 'soporte_posventa', 'despedida']


In [3]:
TIPO_PIEL = {
    "grasa": ["grasa", "oleosa", "aceitosa", "con brillo"],
    "seca": ["seca", "deshidratada", "resecada"],
    "mixta": ["mixta", "combinada"],
    "sensible": ["sensible", "reactiva", "que se irrita facil"],
    "con_acne": ["acne", "granos", "espinillas", "barros"],
}

PRODUCTOS = {
    "limpiador": ["limpiador", "cleanser", "gel limpiador", "jabon facial"],
    "tonico": ["tonico", "toner"],
    "serum": ["serum", "serums"],
    "mascarilla": ["mascarilla", "mask", "sheet mask"],
    "protector_solar": ["protector solar", "bloqueador", "spf", "filtro solar"],
    "crema": ["crema", "moisturizer", "hidratante"],
}

INGREDIENTES = {
    "niacinamida": ["niacinamida", "niacinamide"],
    "acido_hialuronico": ["acido hialuronico", "hialuronico"],
    "centella_asiatica": ["centella asiatica", "centella", "cica"],
    "mucina_caracol": ["mucina de caracol", "baba de caracol", "snail mucin", "caracol"],
    "retinol": ["retinol", "retinoide"],
    "vitamina_c": ["vitamina c", "vit c"],
}

PRECIOS = {"limpiador": 320, "tonico": 380, "serum": 590, "mascarilla": 45,
           "protector_solar": 420, "crema": 480}

RUTINAS = {
    "grasa": ("Dia: limpiador en gel, tonico, serum con niacinamida y protector solar. "
              "Noche: doble limpieza, exfoliante suave 2-3 veces por semana y crema ligera en gel."),
    "seca": ("Dia: limpiador cremoso, tonico hidratante, serum de acido hialuronico, crema rica "
             "y protector solar. Noche: doble limpieza, serum y crema nutritiva."),
    "mixta": ("Dia: limpiador suave, tonico, serum con niacinamida, crema ligera y protector solar. "
              "Noche: doble limpieza, serum hidratante y crema."),
    "sensible": ("Dia: limpiador sin fragancia, tonico calmante con centella asiatica, crema y "
                 "protector solar mineral. Noche: limpieza suave, serum calmante y crema reparadora."),
    "con_acne": ("Dia: limpiador con acido salicilico, tonico, serum con niacinamida y protector "
                 "solar oil-free. Noche: doble limpieza, tratamiento puntual y crema ligera."),
}

INFO_INGREDIENTES = {
    "niacinamida": "Controla la grasa, minimiza poros y pareja el tono de la piel.",
    "acido_hialuronico": "Hidrata profundamente atrayendo agua a la piel; va bien en piel seca o deshidratada.",
    "centella_asiatica": "Calma la irritacion y fortalece la barrera de la piel; muy usada en piel sensible.",
    "mucina_caracol": "Repara, hidrata y da elasticidad; ayuda con marcas leves de acne.",
    "retinol": "Renueva la piel y suaviza lineas finas; se usa de noche y siempre con protector solar al dia siguiente.",
    "vitamina_c": "Ilumina el tono y ayuda con manchas; conviene usarla junto con protector solar porque puede sensibilizar la piel al sol.",
}

print("Entidades y datos de negocio cargados.")


Entidades y datos de negocio cargados.


## 3. Preprocesamiento (normalizar -> tokenizar -> stopwords -> stem)

In [4]:
def normalizar(texto):
    texto = texto.lower()
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join(c for c in texto if unicodedata.category(c) != "Mn")
    return texto


def preprocesar(texto):
    tokens = _tokenizer.tokenize(normalizar(texto))
    return [_stemmer.stem(t) for t in tokens if t not in STOPWORDS_ES]


VOCAB = {nombre: {raiz for ejemplo in datos["ejemplos"] for raiz in preprocesar(ejemplo)}
         for nombre, datos in INTENCIONES.items()}

print("Ejemplo de preprocesamiento:")
print(preprocesar("Que rutina me recomiendan para mi piel grasa?"))
print()
print("Vocabulario de raices para 'consultar_rutina':")
print(sorted(VOCAB["consultar_rutina"]))


Ejemplo de preprocesamiento:
['rutin', 'recomiend', 'piel', 'gras']

Vocabulario de raices para 'consultar_rutina':
['armo', 'buen', 'correct', 'deb', 'gras', 'orden', 'pas', 'piel', 'product', 'recomiend', 'recomiendam', 'rutin', 'sec', 'segu', 'sensibl', 'skincar', 'usar']


## 4. Deteccion de intencion (interseccion de raices)

In [5]:
def detectar_intencion(mensaje, umbral=1):
    raices_msg = set(preprocesar(mensaje))
    puntajes = {nombre: len(raices_msg & vocab) for nombre, vocab in VOCAB.items()}
    mejor = max(puntajes, key=puntajes.get)
    return mejor if puntajes[mejor] >= umbral else "fallback"


print(detectar_intencion("Cuanto cuesta el serum?"))
print(detectar_intencion("Que tal el clima hoy?"))


consultar_precio
saludo


## 5. Extraccion de entidades (diccionarios + regex) y alerta de seguridad

In [6]:
def _buscar_en_diccionario(texto_normalizado, diccionario):
    for clave, sinonimos in diccionario.items():
        if any(re.search(rf"\b{re.escape(s)}\b", texto_normalizado) for s in sinonimos):
            return clave
    return None


def extraer_entidades(mensaje):
    texto = normalizar(mensaje)
    entidades = {}
    tipo_piel = _buscar_en_diccionario(texto, TIPO_PIEL)
    if tipo_piel:
        entidades["tipo_piel"] = tipo_piel
    producto = _buscar_en_diccionario(texto, PRODUCTOS)
    if producto:
        entidades["producto"] = producto
    ingrediente = _buscar_en_diccionario(texto, INGREDIENTES)
    if ingrediente:
        entidades["ingrediente"] = ingrediente
    pedido = re.search(r"\b\d{6}\b", texto)
    if pedido:
        entidades["numero_pedido"] = pedido.group()
    return entidades


def detectar_alerta(mensaje):
    texto = normalizar(mensaje)
    return any(p in texto for p in PALABRAS_ALERTA)


print(extraer_entidades("Mi piel es grasa y quiero saber el precio del serum"))
print(extraer_entidades("Mi pedido 482913 llego incompleto"))
print("alerta:", detectar_alerta("se me irrito la piel"))


{'tipo_piel': 'grasa', 'producto': 'serum'}
{'numero_pedido': '482913'}
alerta: True


## 6. Generacion de respuesta (reglas + fallback + historial)

In [7]:
historial = []

def responder(mensaje):
    if detectar_alerta(mensaje):
        intencion = "alerta_seguridad"
        entidades = extraer_entidades(mensaje)
        respuesta = MENSAJE_ALERTA
    else:
        intencion = detectar_intencion(mensaje)
        entidades = extraer_entidades(mensaje)

        if intencion == "fallback":
            respuesta = FALLBACK
        elif intencion == "consultar_rutina":
            tipo = entidades.get("tipo_piel")
            if tipo in RUTINAS:
                respuesta = f"Para piel {tipo.replace('_', ' ')}: {RUTINAS[tipo]}"
            else:
                respuesta = ("Cuentame, tu piel es grasa, seca, mixta, sensible o con tendencia "
                              "a acne? Asi te paso la rutina adecuada.")
        elif intencion == "consultar_ingredientes":
            ingr = entidades.get("ingrediente")
            if ingr in INFO_INGREDIENTES:
                respuesta = INFO_INGREDIENTES[ingr]
            else:
                respuesta = ("Sobre que ingrediente quieres saber? Por ejemplo niacinamida, "
                              "acido hialuronico, centella asiatica, mucina de caracol, retinol o vitamina C.")
        elif intencion == "consultar_precio":
            prod = entidades.get("producto")
            if prod in PRECIOS:
                respuesta = f"El {prod.replace('_', ' ')} cuesta ${PRECIOS[prod]} MXN."
            else:
                respuesta = ("De que producto quieres el precio? limpiador, tonico, serum, "
                              "mascarilla, protector solar o crema.")
        elif intencion == "consultar_envio":
            respuesta = ("Hacemos envios a todo Mexico: llegan en 3 a 5 dias habiles con guia "
                          "rastreable. El envio cuesta $99 MXN y es gratis en compras mayores a $800 MXN.")
        elif intencion == "soporte_posventa":
            pedido = entidades.get("numero_pedido")
            if pedido:
                respuesta = (f"Ya tengo tu pedido {pedido}. Cuentame que paso (producto incompleto, "
                              "cambio o devolucion) y te doy los siguientes pasos; tienes 5 dias desde "
                              "que lo recibiste para cambios o devoluciones.")
            else:
                respuesta = "Para ayudarte con tu cambio, devolucion o pedido, pasame tu numero de pedido (6 digitos)."
        else:
            respuesta = random.choice(INTENCIONES[intencion]["respuestas"])

    historial.append({"usuario": mensaje, "intencion": intencion, "entidades": entidades, "bot": respuesta})
    return respuesta


print("Bot:", responder("Hola!"))
print("Bot:", responder("Que rutina me recomiendan para piel grasa?"))


Bot: Hola, soy el asistente virtual de Seoul Glow. Puedo ayudarte con rutinas, ingredientes, precios, envios y cambios. En que te ayudo?
Bot: Para piel grasa: Dia: limpiador en gel, tonico, serum con niacinamida y protector solar. Noche: doble limpieza, exfoliante suave 2-3 veces por semana y crema ligera en gel.


## 7. Pruebas: 12 conversaciones distintas
Se cubren las 7 intenciones, la alerta de seguridad y 2 mensajes fuera del alcance del negocio (deben caer en fallback).

In [8]:
historial.clear()  # empezamos un lote de pruebas limpio (sin las llamadas de ejemplo de arriba)

CASOS = [
    ("Hola, buenas tardes!", "saludo"),
    ("Que rutina me recomiendan para piel grasa?", "consultar_rutina"),
    ("Para que sirve la niacinamida?", "consultar_ingredientes"),
    ("Cuanto cuesta el serum?", "consultar_precio"),
    ("Cuanto tarda el envio a Mexico?", "consultar_envio"),
    ("Quiero hacer un cambio de mi pedido 482913", "soporte_posventa"),
    ("Se me irrito la piel con el tonico que compre", "alerta_seguridad"),
    ("Mi piel es sensible, que rutina me recomiendas?", "consultar_rutina"),
    ("El retinol se puede usar de dia?", "consultar_ingredientes"),
    ("Quiero cancelar mi contrato de telefonia", "fallback"),
    ("Cual es la capital de Francia?", "fallback"),
    ("Gracias, hasta luego", "despedida"),
]

for i, (mensaje, _esperada) in enumerate(CASOS, start=1):
    respuesta = responder(mensaje)
    print(f"--- Conversacion {i} ---")
    print("Tu :", mensaje)
    print("Bot:", respuesta)
    print()


--- Conversacion 1 ---
Tu : Hola, buenas tardes!
Bot: Hola, soy el asistente virtual de Seoul Glow. Puedo ayudarte con rutinas, ingredientes, precios, envios y cambios. En que te ayudo?

--- Conversacion 2 ---
Tu : Que rutina me recomiendan para piel grasa?
Bot: Para piel grasa: Dia: limpiador en gel, tonico, serum con niacinamida y protector solar. Noche: doble limpieza, exfoliante suave 2-3 veces por semana y crema ligera en gel.

--- Conversacion 3 ---
Tu : Para que sirve la niacinamida?
Bot: Controla la grasa, minimiza poros y pareja el tono de la piel.

--- Conversacion 4 ---
Tu : Cuanto cuesta el serum?
Bot: El serum cuesta $590 MXN.

--- Conversacion 5 ---
Tu : Cuanto tarda el envio a Mexico?
Bot: Hacemos envios a todo Mexico: llegan en 3 a 5 dias habiles con guia rastreable. El envio cuesta $99 MXN y es gratis en compras mayores a $800 MXN.

--- Conversacion 6 ---
Tu : Quiero hacer un cambio de mi pedido 482913
Bot: Ya tengo tu pedido 482913. Cuentame que paso (producto incompl

## 8. Resultados: precision de intencion y tasa de fallback

In [9]:
df = pd.DataFrame(historial)
df["esperada"] = [e for _, e in CASOS]
df["correcto"] = df["esperada"] == df["intencion"]

precision = df["correcto"].mean()
tasa_fallback = (df["intencion"] == "fallback").mean()

print(df[["usuario", "intencion", "esperada", "correcto"]].to_string(index=False))
print()
print(f"Precision de intencion: {precision:.0%} ({df['correcto'].sum()}/{len(df)} casos)")
print(f"Tasa de fallback: {tasa_fallback:.0%}")

errores = df[~df["correcto"]]
if len(errores):
    print()
    print("Caso(s) con error:")
    print(errores[["usuario", "intencion", "esperada"]].to_string(index=False))


                                        usuario              intencion               esperada  correcto
                           Hola, buenas tardes!                 saludo                 saludo      True
     Que rutina me recomiendan para piel grasa?       consultar_rutina       consultar_rutina      True
                 Para que sirve la niacinamida? consultar_ingredientes consultar_ingredientes      True
                        Cuanto cuesta el serum?       consultar_precio       consultar_precio      True
                Cuanto tarda el envio a Mexico?        consultar_envio        consultar_envio      True
     Quiero hacer un cambio de mi pedido 482913       soporte_posventa       soporte_posventa      True
  Se me irrito la piel con el tonico que compre       alerta_seguridad       alerta_seguridad      True
Mi piel es sensible, que rutina me recomiendas?       consultar_rutina       consultar_rutina      True
               El retinol se puede usar de dia? consultar_ingred

## 9. Limitaciones observadas y mejoras para el Avance 2
- El unico error del lote fue *"Quiero cancelar mi contrato de telefonia"*, clasificado como `soporte_posventa` en vez de `fallback`: la raiz `cancel` aparece en los ejemplos de `soporte_posventa` ("quiero cancelar mi pedido"), y como el bot solo cuenta coincidencias de raiz sin revisar el resto de la frase, "telefonia" (fuera del catalogo de Seoul Glow) no bastó para descartar esa interseccion.
- Limitacion general: al ser reglas por interseccion de palabras, el bot no entiende negaciones ni contexto (por ejemplo, no distingue "quiero cancelar" de "ya no quiero cancelar").
- Mejora propuesta para el Avance 2: agregar una lista de palabras que excluyan una intencion aunque haya coincidencia de raiz (por ejemplo, si aparece "telefonia", "internet" u otro servicio que Seoul Glow no vende, forzar fallback), y subir el umbral minimo de coincidencias de 1 a 2 para reducir falsos positivos.